# Categorías de películas desde su sinopsis: Evaluación

Predecir varias categorías de catálogo para una película desde su sinopsis.

F1 macro/micro y coincidencia exacta sobre categorías conocidas de train. Las asignaciones de categorías desconocidas se cuentan por separado. No todas las categorías de catálogo son géneros estrictos.

Consulta la [metodología](../docs/metodologia.md). Los reportes corresponden a la última ejecución de cada etapa.


In [1]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("netflix_genre_classification")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from netflix_genre_classification.alerts import configurar_notebook, mostrar_alertas
configurar_notebook()


In [2]:
from netflix_genre_classification.flow import run_stage
metricas, comentario = run_stage(PROJECT_ROOT, "evaluacion")
display(Markdown(comentario))
from netflix_genre_classification.presentation import display_evaluation
display_evaluation(PROJECT_ROOT, metricas)


# Categorías de películas desde su sinopsis: evaluacion

En **1,222 textos de test**, el modelo obtiene **F1 macro 0.1716** y supera el baseline (0.0000). Hay **0 asignaciones de categorías fuera del vocabulario de train** y **8 categorías presentes en test con recall cero**. Las métricas solo cubren categorías conocidas; deben interpretarse junto con esos conteos y el detalle por categoría. Superar el baseline no acredita que el modelo detecte adecuadamente todas las categorías. Las entradas contienen exclusivamente sinopsis.


,baseline,modelo
casos,1222.0,1222.000000
exact_match_etiquetas_conocidas,0.0,0.108020
f1_macro,0.0,0.171649
etiquetas_fuera_vocabulario_train,0.0,0.000000
f1_micro,0.0,0.435125


,precision,recall,f1-score,support
Action & Adventure,0.833333,0.142857,0.243902,175.0
Anime Features,0.000000,0.000000,0.000000,21.0
Children & Family Movies,1.000000,0.105691,0.191176,123.0
Classic Movies,0.000000,0.000000,0.000000,20.0
Comedies,0.745283,0.233728,0.355856,338.0
Cult Movies,0.000000,0.000000,0.000000,15.0
Documentaries,0.986667,0.427746,0.596774,173.0
Dramas,0.706349,0.522505,0.600675,511.0
Faith & Spirituality,0.000000,0.000000,0.000000,14.0
Horror Movies,1.000000,0.013699,0.027027,73.0


## Alertas

Avisos agrupados con descripción y acción sugerida. Los errores no se ocultan; no registrar warnings no garantiza que el análisis sea válido.

In [ ]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/05_evaluacion.json")
from netflix_genre_classification.informe import generar_informe
generar_informe(PROJECT_ROOT)
